In [ ]:
from pathlib import Path

import polars as pl


from climate_attitudes.extract.dataset import ClimateAttitudesDataset

ASSETS_DIR = Path("/Users/henry/data/msc_thesis/climate-attitudes")

In [ ]:
data = ClimateAttitudesDataset(ASSETS_DIR)

### "Born in the USA"

Item `dem_US` asks participants whether they were born in the USA (Yes: 1, No: 0). This item is always presented to new participants, but 
in Wave 5 was also presented to returning participants. We expect that participants who answer this question multiple times should provide 
the same response each time. 

In [ ]:
data.participant

In [ ]:
# Select participants who respond to wave 5, but joined earlier
wave_5_repeating_pids = data.participant.filter(
    pl.col("wave_joined") < 5,
    pl.col("wave_5"),
).select(pl.col("participant_id").unique().sort())

Count how often each combination of non-null responses occurs

In [ ]:
(
    wave_5_repeating_pids.join(
        data.question_response.filter(pl.col("item_name") == "dem_US"),
        on="participant_id",
        how="left",
    )
    .with_columns(pl.col("response").cast(pl.Int64))
    .filter(pl.col("response").is_not_null())
    .group_by("participant_id")
    .agg("response")
    .select(pl.col("response").value_counts())
)

Most responses are either (1,1) or (0,0) as expected. However, 

1. Several participants change their response between waves, as indicated by the non-zero counts for (1,0) and (0,1).

2. In two cases participants only have one recorded response, despite `null` not being an option for this question.

3. In several cases participants have three recorded responses, as indicated by the non-zero counts for (1,1,1), (0,0,0), and (0,1,1).

#### Only one response

This may be due to a missing PID from participants' first response (in which case they would not be presented the question again until W5). Alternatively it may be possible for 
participants to skip questions, or there may have been a data error.

#### Three responses

This is likely either a data error or a survey logic error. 

#### Variable response

In [ ]:
variable_response_pids = (
    (
        wave_5_repeating_pids.join(
            data.question_response.filter(pl.col("item_name") == "dem_US"),
            on="participant_id",
            how="left",
        )
        .with_columns(pl.col("response").cast(pl.Int64))
        .filter(pl.col("response").is_not_null())
        .group_by("participant_id")
        .agg("response")
        .filter(pl.col("response").is_in([[0, 1], [1, 0]]))
        .select("participant_id")
    )
    .to_series()
    .implode()
)

In [ ]:
(
    data.question_response.filter(
        pl.col("participant_id").is_in(variable_response_pids),
        pl.col("item_name").str.starts_with("dem_"),
    )
    .select("participant_id", "wave", "item_name", pl.col("response"))
    .pivot("item_name", values="response")
)